In [40]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
# from ydata_profiling import ProfileReport
# import missingno as msno

df = pd.read_csv('DNN-EdgeIIoT-dataset.csv')


## Functions

In [41]:
def get_type_missing(df):
    df_types = pd.DataFrame()
    df_types['data_type'] = df.dtypes
    df_types['missing_values'] = df.isnull().sum()
    return df_types.sort_values(by='missing_values', ascending=False)

## Checking Health of Data

In [3]:
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

In [42]:
df.sample(10)

,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
578863,2021 19:37:28.293967000,0,0,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
102181,2021 13:30:48.840232000,192.168.0.128,192.168.0.101,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
62384,2021 12:49:16.854670000,192.168.0.128,192.168.0.101,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
947734,2021 16:53:54.123032000,192.168.0.101,192.168.0.128,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
337961,2021 17:37:48.469530000,192.168.0.128,192.168.0.101,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
843833,2021 22:02:18.568560000,192.168.0.128,192.168.0.101,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
1296791,2021 16:28:22.304947000,192.168.0.101,192.168.0.128,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
755280,2021 20:29:16.086004000,192.168.0.101,192.168.0.128,0,0.0,0.0,0,0.0,0.0,0.0,...,4.0,MQTT,0,0.0,4.0,0.0,0.0,0.0,0,Normal
2071132,11.165.156.210,192.168.0.128,0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,DDoS_UDP
2193067,2021 23:23:20.192916000,171.202.72.7,192.168.0.128,0,0.0,0.0,0,49691.0,56558.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,DDoS_ICMP


In [43]:
drop_columns = [
          # Source IP - will vary, doesn't indicate attack type
    "ip.dst_host",          # Destination IP - same reason
    "arp.src.proto_ipv4",   # ARP protocol detail - redundant
    "arp.dst.proto_ipv4",   # ARP protocol detail - redundant
    "http.file_data",       # Large binary data - too much memory
    "http.request.full_uri",# Full URL - too specific, causes overfitting
    "icmp.transmit_timestamp", # Timestamp variant - not useful
    "http.request.uri.query", # Query parameters - too specific
    "tcp.options",          # Low-level TCP detail - rarely discriminative
    "tcp.payload",          # Raw binary - too large
    "udp.port",             # Port info - not generalizable
    "mqtt.msg"              # Message content - too specific
]
 
# Only drop columns that actually exist in the dataset
drop_columns = [col for col in drop_columns if col in df.columns]
 


 

In [44]:
drop_columns

['ip.dst_host',
 'arp.src.proto_ipv4',
 'arp.dst.proto_ipv4',
 'http.file_data',
 'http.request.full_uri',
 'icmp.transmit_timestamp',
 'http.request.uri.query',
 'tcp.options',
 'tcp.payload',
 'udp.port',
 'mqtt.msg']

In [45]:
missing = df.isna().sum()
print(f"{missing}")
print("No missing values")

frame.time            0
ip.src_host           0
ip.dst_host           0
arp.dst.proto_ipv4    0
arp.opcode            0
                     ..
mbtcp.len             0
mbtcp.trans_id        0
mbtcp.unit_id         0
Attack_label          0
Attack_type           0
Length: 63, dtype: int64
No missing values


In [46]:
duplicates = df.duplicated().sum()
print(duplicates)

df.drop_duplicates(subset=None, keep="first", inplace=True)
 
# duplicates = df.duplicated().sum()


815


In [47]:
get_type_missing(df)

,data_type,missing_values
frame.time,str,0
ip.src_host,str,0
ip.dst_host,object,0
arp.dst.proto_ipv4,object,0
arp.opcode,float64,0
...,...,...
mbtcp.len,float64,0
mbtcp.trans_id,float64,0
mbtcp.unit_id,float64,0
Attack_label,int64,0


In [48]:
pd.DataFrame(df.value_counts())

,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,count
frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,icmp.unused,http.file_data,http.content_length,http.request.uri.query,http.request.method,http.referer,http.request.full_uri,http.request.version,http.response,http.tls_port,tcp.ack,tcp.ack_raw,tcp.checksum,tcp.connection.fin,tcp.connection.rst,tcp.connection.syn,tcp.connection.synack,tcp.dstport,tcp.flags,tcp.flags.ack,tcp.len,tcp.options,tcp.payload,tcp.seq,tcp.srcport,udp.port,udp.stream,udp.time_delta,dns.qry.name,dns.qry.name.len,dns.qry.qu,dns.qry.type,dns.retransmission,dns.retransmit_request,dns.retransmit_request_in,mqtt.conack.flags,mqtt.conflag.cleansess,mqtt.conflags,mqtt.hdrflags,mqtt.len,mqtt.msg_decoded_as,mqtt.msg,mqtt.msgtype,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type,
2021 11:44:10.081753000,192.168.0.128,192.168.0.101,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.153346e+09,47892.0,0.0,0.0,0.0,1.0,64855.0,18.0,1.0,0.0,020405b40101040201030307,0,0.0,1883.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal,1
2021 11:44:10.162218000,192.168.0.101,192.168.0.128,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.447945e+08,16077.0,0.0,0.0,0.0,0.0,1883.0,24.0,1.0,14.0,0,100c00044d5154540402003c0000,1.0,64855.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0,1.0,2.0,16.0,12.0,0.0,0,1.0,4.0,MQTT,0,0.0,4.0,0.0,0.0,0.0,0,Normal,1
2021 11:44:10.162271000,192.168.0.128,192.168.0.101,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15.0,3.153346e+09,62675.0,0.0,0.0,0.0,0.0,64855.0,16.0,1.0,0.0,0,0,1.0,1883.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal,1
2021 11:44:10.162641000,192.168.0.128,192.168.0.101,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15.0,3.153346e+09,54469.0,0.0,0.0,0.0,0.0,64855.0,24.0,1.0,4.0,0,20020000,1.0,1883.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0x00000000,0.0,0.0,32.0,2.0,0.0,0,2.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal,1
2021 11:44:10.166132000,192.168.0.101,192.168.0.128,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,3.447945e+08,53812.0,0.0,0.0,0.0,0.0,1883.0,24.0,1.0,41.0,0,3027001854656d70657261747572655f616e645f48756d696469747932342e36382037362e34320d0a,15.0,64855.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,48.0,39.0,0.0,32342e36382037362e34320d0a,3.0,0.0,0,Temperature_and_Humidity,24.0,0.0,0.0,0.0,0.0,0,Normal,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021 23:24:32.816050000,166.75.162.225,192.168.0.128,0,0.0,0.0,0,31814.0,45620.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,DDoS_ICMP,1
2021 23:24:32.816595000,70.162.34.183,192.168.0.128,0,0.0,0.0,0,27718.0,45636.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,DDoS_ICMP,1
2021 23:24:32.818043000,40.13.95.244,192.168.0.128,0,0.0,0.0,0,18502.0,45672.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,DDoS_ICMP,1


In [49]:
df.describe(include='object')

,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.src.proto_ipv4,http.file_data,http.request.uri.query,http.request.method,http.referer,http.request.full_uri,http.request.version,tcp.options,tcp.payload,tcp.srcport,dns.qry.name.len,mqtt.conack.flags,mqtt.msg,mqtt.protoname,mqtt.topic,Attack_type
count,2218386,2218386,2218386,2218386,2218386,2218386.0,2218386.0,2218386.0,2218386.0,2218386.0,2218386.0,2218386,2218386,2218386.0,2218386,2218386,2218386,2218386,2218386,2218386
unique,2206364,137167,52426,10,11,2397.0,5527.0,10.0,6.0,11409.0,14.0,242566,274927,63526.0,11,14,137,4,4,15
top,192.168.0.128,192.168.0.128,192.168.0.128,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1883.0,0,0,0,0,0,Normal
freq,11533,962129,958578,1685334,1591684,1989010.0,2087314.0,1924288.0,2103698.0,1924288.0,1924288.0,1453980,1287079,661308.0,1973440,1532586,1532627,1532608,1532627,1615643


In [61]:
# profile = ProfileReport(df, title="Profiling Report")
# profile.to_file('Data Analysis.html')
# This will generate an interactive report for EDA. Useful function for quick EDA
df.drop(columns=drop_columns,inplace=True)

KeyError: "['ip.dst_host', 'arp.src.proto_ipv4', 'arp.dst.proto_ipv4', 'http.file_data', 'http.request.full_uri', 'icmp.transmit_timestamp', 'http.request.uri.query', 'tcp.options', 'tcp.payload', 'udp.port', 'mqtt.msg'] not found in axis"

In [62]:
new_colums=['dns.qry.name.len', 'mqtt.conack.flags', 'mqtt.protoname','mqtt.topic']
obj_columns=[]
num_columns=[]
for col in df.columns:
    dtype = df[col].dtype
    unique = df[col].nunique()
    sample = df[col].iloc[0]
    if(unique)==1:
        new_colums.append(col)
        
        
    
    if dtype == 'object':
        print(f"  {col:30s} | Type: {str(dtype):10s} | Unique: {unique:6d} | Sample: {str(sample)[:40]}")
        obj_columns.append(col)
    else:
        num_columns.append(col)
        print(f"{col:30s} | Type: {str(dtype):10s} | Unique: {unique:6d} Min: {df[col].min()}, Max: {df[col].max()}")

try:
    df.drop(columns=new_colums,inplace=True)
except:
 print("hi")
    


frame.time                     | Type: str        | Unique:  99514 Min:  2021 00:00:24.142728000 , Max: 99.82.134.219
ip.src_host                    | Type: str        | Unique:   6166 Min: 0, Max: 99.95.152.164
arp.opcode                     | Type: float64    | Unique:      3 Min: 0.0, Max: 2.0
arp.hw.size                    | Type: float64    | Unique:      2 Min: 0.0, Max: 6.0
icmp.checksum                  | Type: float64    | Unique:   5093 Min: 0.0, Max: 65528.0
icmp.seq_le                    | Type: float64    | Unique:   5507 Min: 0.0, Max: 65468.0
http.content_length            | Type: float64    | Unique:     29 Min: 0.0, Max: 1824.0
  http.request.method            | Type: object     | Unique:      8 | Sample: 0.0
  http.referer                   | Type: object     | Unique:      5 | Sample: 0.0
  http.request.version           | Type: object     | Unique:      6 | Sample: 0.0
http.response                  | Type: float64    | Unique:      2 Min: 0.0, Max: 1.0
tcp.ack     

In [52]:
print("working columns",len(df.columns))

working columns 44


In [63]:
df.columns

Index(['frame.time', 'ip.src_host', 'arp.opcode', 'arp.hw.size',
       'icmp.checksum', 'icmp.seq_le', 'http.content_length',
       'http.request.method', 'http.referer', 'http.request.version',
       'http.response', 'tcp.ack', 'tcp.ack_raw', 'tcp.checksum',
       'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn',
       'tcp.connection.synack', 'tcp.dstport', 'tcp.flags', 'tcp.flags.ack',
       'tcp.len', 'tcp.seq', 'tcp.srcport', 'udp.stream', 'udp.time_delta',
       'dns.qry.name', 'dns.qry.qu', 'dns.retransmission',
       'dns.retransmit_request', 'dns.retransmit_request_in',
       'mqtt.conflag.cleansess', 'mqtt.conflags', 'mqtt.hdrflags', 'mqtt.len',
       'mqtt.msgtype', 'mqtt.proto_len', 'mqtt.topic_len', 'mqtt.ver',
       'mbtcp.len', 'mbtcp.trans_id', 'mbtcp.unit_id', 'Attack_label',
       'Attack_type'],
      dtype='str')

In [64]:
df.select_dtypes(include=['object']).columns.tolist()
df = df[pd.to_numeric(df["tcp.srcport"], errors="coerce").notna()]
df["tcp.srcport"] = df["tcp.srcport"].astype(float)

In [65]:
categorical_cols = [
    col for col in df.select_dtypes(include=['object']).columns.tolist()
    if col not in ["frame.time", 'ip.src_host']
]   
print(f"Categorical columns (need encoding): {len(categorical_cols)}")
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
 
# Remove target variable from numerical
if 'Attack_type' in categorical_cols:
    categorical_cols.remove('Attack_type')
 
print(f"\nCategorical columns (need encoding): {len(categorical_cols)}")
for col in categorical_cols:
    unique_vals = df[col].nunique()
    samples = df[col].unique()[:3]
    print(f"  • {col:30s} ({unique_vals} unique values)")
    print(f"    Examples: {samples}")

Categorical columns (need encoding): 4

Categorical columns (need encoding): 3
  • http.request.method            (8 unique values)
    Examples: [0.0 '0' '0.0']
  • http.referer                   (5 unique values)
    Examples: [0.0 '0' '0.0']
  • http.request.version           (6 unique values)
    Examples: [0.0 '0' '0.0']


In [60]:
df=df_bck
df_bck=df
df= df.sample(n=100000, random_state=42)

In [66]:
categorical_cols

['http.request.method', 'http.referer', 'http.request.version']

In [67]:

def encode_text_dummy(df, name):
    """One-hot encode categorical column"""
    if name in df.columns:
        dummies = pd.get_dummies(df[name], prefix=name)
        df = pd.concat([df, dummies], axis=1)
        df.drop(name, axis=1, inplace=True)
        return df, True
    return df, False
 
encoded_count = 0
for col in categorical_cols:
    df, success = encode_text_dummy(df, col)
    if success:
        encoded_count += 1
        print(f"  ✓ Encoded: {col}")
 
print(f"\n✓ Total columns after encoding: {df.shape[1]}")

  ✓ Encoded: http.request.method
  ✓ Encoded: http.referer
  ✓ Encoded: http.request.version

✓ Total columns after encoding: 60


In [69]:
df


,frame.time,ip.src_host,arp.opcode,arp.hw.size,icmp.checksum,icmp.seq_le,http.content_length,http.response,tcp.ack,tcp.ack_raw,...,http.referer_() { _; } >_[$($())] { echo 93e4r0-CVE-2014-6278: true; echo;echo; },http.referer_0,http.referer_0.0,http.referer_127.0.0.1,http.request.version_0.0,http.request.version_0,http.request.version_0.0,http.request.version_HTTP/1.0,http.request.version_HTTP/1.1,http.request.version_name=a><input name=i value=XSS>&lt;script>alert('Vulnerable')</script> HTTP/1.1
300987,2021 16:58:57.546116000,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,6.0,1.331938e+09,...,False,False,False,False,True,False,False,False,False,False
1973266,2021 18:58:47.076449000,192.168.0.170,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,...,False,True,False,False,False,True,False,False,False,False
713532,2021 20:03:12.625068000,0,0.0,0.0,0.0,0.0,0.0,0.0,188753094.0,1.327605e+09,...,False,False,False,False,True,False,False,False,False,False
1115407,2021 20:13:31.112839000,192.168.0.128,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,...,False,False,False,False,True,False,False,False,False,False
1043632,2021 18:33:35.561355000,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,6.0,8.016943e+08,...,False,False,False,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1409518,2021 18:26:46.532597000,192.168.0.128,0.0,0.0,0.0,0.0,0.0,0.0,56.0,1.061200e+09,...,False,False,False,False,True,False,False,False,False,False
680839,2021 19:57:28.326446000,0,0.0,0.0,0.0,0.0,0.0,0.0,612074.0,2.372237e+09,...,False,False,False,False,True,False,False,False,False,False
714537,2021 20:03:23.058305000,0,0.0,0.0,0.0,0.0,0.0,0.0,688853.0,2.372314e+09,...,False,False,False,False,True,False,False,False,False,False
1101122,2021 19:58:30.536433000,192.168.0.128,0.0,0.0,0.0,0.0,0.0,0.0,59.0,2.337598e+09,...,False,False,False,False,True,False,False,False,False,False


In [32]:
df

,arp.opcode,arp.hw.size,icmp.checksum,icmp.seq_le,http.content_length,http.response,tcp.ack,tcp.ack_raw,tcp.checksum,tcp.connection.fin,...,http.referer_() { _; } >_[$($())] { echo 93e4r0-CVE-2014-6278: true; echo;echo; },http.referer_0,http.referer_0.0,http.referer_127.0.0.1,http.request.version_0.0,http.request.version_0,http.request.version_0.0,http.request.version_> HTTP/1.1,http.request.version_HTTP/1.0,http.request.version_HTTP/1.1
593348,0.0,0.0,0.0,0.0,0.0,0.0,413519.0,2.372038e+09,38287.0,0.0,...,False,False,False,False,True,False,False,False,False,False
589711,0.0,0.0,0.0,0.0,0.0,0.0,404286.0,2.372029e+09,27133.0,0.0,...,False,False,False,False,True,False,False,False,False,False
994049,0.0,0.0,0.0,0.0,0.0,0.0,59.0,3.138946e+09,231.0,0.0,...,False,False,False,False,True,False,False,False,False,False
558827,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,44608.0,0.0,...,False,False,False,False,True,False,False,False,False,False
153462,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.452001e+09,4820.0,0.0,...,False,False,False,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1450325,0.0,0.0,0.0,0.0,0.0,0.0,1933722.0,1.142821e+09,24590.0,0.0,...,False,False,False,False,True,False,False,False,False,False
195449,0.0,0.0,0.0,0.0,0.0,0.0,6.0,3.854458e+09,24435.0,0.0,...,False,False,False,False,True,False,False,False,False,False
1189524,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,34482.0,0.0,...,False,False,False,False,True,False,False,False,False,False
874418,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.571767e+08,30263.0,0.0,...,False,False,False,False,True,False,False,False,False,False


In [70]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
import pickle
import warnings
warnings.filterwarnings('ignore')
 
# TensorFlow/Keras imports
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

In [71]:
df['frame.time'] = pd.to_datetime(df['frame.time'], errors='coerce')
df = df.sort_values(['ip.src_host', 'frame.time'])

In [91]:
exclude_cols = ['frame.time', 'ip.src_host', 'Attack_type',"Attack_label"]
feature_cols = [col for col in   df.columns if col not in exclude_cols]

In [92]:
X_all = df[feature_cols]
y_all = df['Attack_type']

In [93]:
for i in feature_cols:
    print(f"{i}",end=", ")

arp.opcode, arp.hw.size, icmp.checksum, icmp.seq_le, http.content_length, http.response, tcp.ack, tcp.ack_raw, tcp.checksum, tcp.connection.fin, tcp.connection.rst, tcp.connection.syn, tcp.connection.synack, tcp.dstport, tcp.flags, tcp.flags.ack, tcp.len, tcp.seq, tcp.srcport, udp.stream, udp.time_delta, dns.qry.name, dns.qry.qu, dns.retransmission, dns.retransmit_request, dns.retransmit_request_in, mqtt.conflag.cleansess, mqtt.conflags, mqtt.hdrflags, mqtt.len, mqtt.msgtype, mqtt.proto_len, mqtt.topic_len, mqtt.ver, mbtcp.len, mbtcp.trans_id, mbtcp.unit_id, http.request.method_0.0, http.request.method_0, http.request.method_0.0, http.request.method_GET, http.request.method_POST, http.request.method_PROPFIND, http.request.method_PUT, http.request.method_TRACE, http.referer_0.0, http.referer_() { _; } >_[$($())] { echo 93e4r0-CVE-2014-6278: true; echo;echo; }, http.referer_0, http.referer_0.0, http.referer_127.0.0.1, http.request.version_0.0, http.request.version_0, http.request.version

In [94]:
SEQ_LEN = 10

def create_sequences(df, feature_cols, seq_len=10):
    X_seq, y_seq = [], []

    for ip, group in df.groupby('ip.src_host'):
        group = group.sort_values('frame.time')

        X = group[feature_cols].values
        y = group['Attack_type'].values

        for i in range(len(group) - seq_len):
            X_seq.append(X[i:i+seq_len])
            y_seq.append(y[i+seq_len])

    return np.array(X_seq), np.array(y_seq)

In [95]:
X_seq, y_seq = create_sequences(df, feature_cols, seq_len=10)
split_ratio = 0.8
split_index = int(len(X_seq) * split_ratio)

X_train = X_seq[:split_index]
X_test  = X_seq[split_index:]

y_train = y_seq[:split_index]
y_test  = y_seq[split_index:]

In [96]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

le = LabelEncoder()

# FIT ON FULL DATA FIRST
y_encoded_all = le.fit_transform(y_seq)

In [97]:
split = int(len(X_seq) * 0.8)

X_train, X_test = X_seq[:split], X_seq[split:]
y_train, y_test = y_encoded_all[:split], y_encoded_all[split:]

In [98]:
y_train_cat = to_categorical(y_train)
y_test_cat = to_categorical(y_test)

In [99]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

timesteps = X_train.shape[1]
features = X_train.shape[2]

model = Sequential()

# LSTM Layer 1
model.add(LSTM(64, return_sequences=True, input_shape=(timesteps, features)))
model.add(Dropout(0.3))

# LSTM Layer 2
model.add(LSTM(32, return_sequences=False))
model.add(Dropout(0.3))

# Dense Layers
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))

# Output Layer
model.add(Dense(num_classes, activation='softmax'))

# Compile
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 10, 64)         │        32,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 10, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 15)             │           495 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 46,479 (181.56 KB)

 Trainable params: 46,479 (181.56 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_test, y_test_cat),   
    epochs=50,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

Epoch 1/50
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - accuracy: 0.9017 - loss: 0.5352 - val_accuracy: 0.2379 - val_loss: 7.2146 - learning_rate: 0.0010
Epoch 2/50
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 20s 17ms/step - accuracy: 0.9120 - loss: 0.4298 - val_accuracy: 0.2379 - val_loss: 7.4892 - learning_rate: 0.0010
Epoch 3/50
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 21s 17ms/step - accuracy: 0.9121 - loss: 0.4193 - val_accuracy: 0.2379 - val_loss: 8.2892 - learning_rate: 0.0010
Epoch 4/50
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 19s 16ms/step - accuracy: 0.9121 - loss: 0.4130 - val_accuracy: 0.2379 - val_loss: 10.2250 - learning_rate: 0.0010
Epoch 5/50
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 19s 16ms/step - accuracy: 0.9121 - loss: 0.4108 - val_accuracy: 0.2379 - val_loss: 9.1787 - learning_rate: 0.0010
Epoch 6/50
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 20s 17ms/step - accuracy: 0.9122 - loss: 0.4084 - val_accuracy: 0.2379 - val_loss: 10.0593 - learning_rate: 0.0010
Epoch 7/50
1173/1173 ━━━━━━━━━━━━━━━━━━━━ 22s 18ms/step - accu

In [101]:
X_train = X_train.astype('float32')
X_test  = X_test.astype('float32')

In [36]:
X = df.drop('Attack_type', axis=1).values
y = df['Attack_type'].values

In [37]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_categorical = to_categorical(y_encoded)
 
num_classes = len(label_encoder.classes_)
 

In [38]:
print(df['icmp.checksum'].value_counts())
print(df['Attack_label'].value_counts())

icmp.checksum
0.0        94734
28353.0        3
42976.0        3
54880.0        3
29276.0        3
           ...  
58636.0        1
37701.0        1
43059.0        1
23078.0        1
26798.0        1
Name: count, Length: 5052, dtype: int64
Attack_label
0    72774
1    27208
Name: count, dtype: int64


In [39]:
num_classes = len(label_encoder.classes_)
 
print(f"✓ Target encoded")
print(f"  Number of classes: {num_classes}")
print(f"  Classes: {label_encoder.classes_}")
 

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
 

✓ Target encoded
  Number of classes: 15
  Classes: ['Backdoor' 'DDoS_HTTP' 'DDoS_ICMP' 'DDoS_TCP' 'DDoS_UDP' 'Fingerprinting'
 'MITM' 'Normal' 'Password' 'Port_Scanning' 'Ransomware' 'SQL_injection'
 'Uploading' 'Vulnerability_scanner' 'XSS']


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_categorical, test_size=0.2, random_state=42, stratify=y_encoded
)
 
print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")
 
# Reshape for LSTM: (samples, timesteps, features)
# Option 1: Treat each sample as 1 timestep
timesteps = 1
X_train_lstm = X_train.reshape((X_train.shape[0], timesteps, X_train.shape[1]))
X_test_lstm = X_test.reshape((X_test.shape[0], timesteps, X_test.shape[1]))
 
print(f"X_train LSTM shape: {X_train_lstm.shape}")
print(f"X_test LSTM shape: {X_test_lstm.shape}")
 
# Alternative: Create sequences with multiple timesteps
def create_sequences(data, seq_length=10):
    """Create sequences for LSTM"""
    X_seq = []
    for i in range(len(data) - seq_length + 1):
        X_seq.append(data[i:i+seq_length])
    return np.array(X_seq)
 
print("\n✓ Data preparation complete!")
 
# STEP 11: Build LSTM Model
# ============================================================================
print("\n" + "="*60)
print("BUILDING LSTM MODEL")
print("="*60)
 
model = Sequential([
    # LSTM Layer 1
    LSTM(64, activation='relu', input_shape=(timesteps, X_train.shape[1]), return_sequences=True),
    Dropout(0.2),
    
    # LSTM Layer 2
    LSTM(32, activation='relu', return_sequences=False),
    Dropout(0.2),
    
    # Dense Layers
    Dense(16, activation='relu'),
    Dropout(0.2),
    
    # Output Layer
    Dense(num_classes, activation='softmax')
])
 
# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
 
print(model.summary())
 
# STEP 12: Train the Model
# ============================================================================
print("\n" + "="*60)
print("TRAINING LSTM MODEL")
print("="*60)
 
# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7)
 
# Train
history = model.fit(
    X_train_lstm, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)
 
print("\n✓ Training complete!")
 

In [ ]:
df.columns

In [ ]:
# df.drop(columns=['tcp.flags','icmp.checksum', 'tcp.checksum', 'tcp.connection.synack',
#                 'tcp.seq'], inplace=True)

In [ ]:
# # Group by 'http.response' and 'Attack_label' and count occurrences
# attack_response_counts = df.groupby(['tcp.ack_', 'Attack_label']).size().unstack(fill_value=0)

# print(attack_response_counts)


In [ ]:
# df.columns

In [ ]:
plt.figure(figsize=(15, 6))
sns.countplot(data=df[df['Attack_label'] == 1], x='Attack_label', hue='Attack_type', 
              edgecolor='black', linewidth=1, palette='muted')
plt.title('Attack Label vs Attack Type', fontsize=20)
plt.show()

In [ ]:
import plotly.express as px

fig = px.pie(df, names='Attack_label', title='Distribution of Attack Labels')
fig.show()


In [ ]:
fig = px.pie(df, names='Attack_type', title='Distribution of Attack Type')
fig.show()


In [ ]:
df['Attack_label'].value_counts()

- class imbalance issue - this can cause the machine learning model to result in biased results

- Problem: if we use a machine learning model to predict the Attack label, it could predict it as 0.1, 0.2 or 0.99 which is not a valid Attack label
- Solution: Label Encoder

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['Attack_label'] = le.fit_transform(df['Attack_label'])

df['Attack_label'].value_counts()

- seperate X and y variables (independent and dependent variables)

In [ ]:
X = df.drop(['Attack_label', 'Attack_type'], axis=1)
y_label = df['Attack_label']
y_type = df['Attack_type']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train_label, y_test_label = train_test_split(X, y_label, test_size=0.2, random_state=42)

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy=1, random_state=42)
X_train_resampled, y_train_label_resampled = smote.fit_resample(X_train, y_train_label)

In [ ]:
print(y_train_label.value_counts())
print(y_train_label_resampled.value_counts())

## Predicting Labels -> Attack or Not

## Logistic Regression